In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
dbutils.widgets.text('catalog','fmcg','Catalog')

catalog = dbutils.widgets.get('catalog')

print(catalog)
print(gold_schema)

fmcg
gold


In [0]:
%run /Workspace/Data_Ingestion_Atlikon/utilities

In [0]:
df_customers = spark.sql(f"select* from {catalog}.{gold_schema}.dim_customers")
df_sb_dim_customers = spark.sql(f"select customer_code, customer, market, platform, channel from {catalog}.{gold_schema}.sb_dim_customers")

In [0]:
display(df_customers.limit(5))
display(df_sb_dim_customers.limit(5))

customer_code,customer,market,platform,channel
70002017,Sporty Essentials,India,Brick & Mortar,Retailer
70002018,PlayMore Sports,India,Brick & Mortar,Retailer
90002001,SportsHub,India,E-Commerce,Retailer
90002002,PowerPlay Sports,India,Brick & Mortar,Retailer
90002003,Winning Gear,India,Brick & Mortar,Retailer


customer_code,customer,market,platform,channel
789503,Peak Performance Store-New Delhi,India,Sports Bar,Acquisition
789420,Zenathlete Foods-Bangalore,India,Sports Bar,Acquisition
789703,Staminax Store-New Delhi,India,Sports Bar,Acquisition
789621,Eliteathlete Nutrition-Hyderabad,India,Sports Bar,Acquisition
789101,Endurance Foods-Bangalore,India,Sports Bar,Acquisition


### Perform transformation on customers table as is

In [0]:
df_customers_unmatched = df_customers.join(df_sb_dim_customers, on= "customer_code", how= "left_anti")

df_merged = df_customers_unmatched.unionByName(df_sb_dim_customers)

In [0]:
df_merged.write.format("delta")\
    .mode("overwrite")\
    .option("delta.enableChangeDataFeed", "true")\
    .option("overwriteSchema", "true")\
    .saveAsTable(f"{catalog}.{gold_schema}.dim_customers")

### Only select required columns and upsert in target table with more priority given to source table for Products data

In [0]:
delta_table = DeltaTable.forName(spark, "fmcg.gold.dim_products")
df_sb_dim_products = spark.sql(f"select product_code, division, category, product, variant from {catalog}.{gold_schema}.sb_dim_products")

df_sb_dim_products.printSchema()
display(df_sb_dim_products.limit(5))

root
 |-- product_code: string (nullable = true)
 |-- division: string (nullable = true)
 |-- category: string (nullable = true)
 |-- product: string (nullable = true)
 |-- variant: string (nullable = true)



product_code,division,category,product,variant
fe5a8036be4b9a787b7c0ae013fc752a8cfb6c55a2f7b2fd152a6380925e9c49,Dairy & Recovery,Recovery Dairy,SportsBar Greek Yogurt Pro Vanilla (120g),120g
451f7167b28a25bde73995910e31c07dfa26411f1db47847f19e16747effbdaa,Hydration & Electrolytes,Electrolyte Mix,SportsBar Electrolyte Mix Lemon-Lime (5 Sachets),5 Sachets
e92c739a8d78cd6cbe954648c2f9dd75ed61fcfd99b03e10dca65c3082d0728e,Nutrition Bars,Energy Bars,SportsBar Energy Bar Choco Fudge (40g),40g
3cab59f05924285270313afcfe40a08983bb03dd88f432e34fc6336914c14345,Breakfast Foods,Granola & Cereals,SportsBar Granola Crunch Honey Almond (400g),400g
77b6f538a9d0e0cf845db5c2cbecec46fdd30303b501e06f64baf1d4dc0e66f9,Dairy & Recovery,Recovery Dairy,SportsBar Greek Yogurt Pro Vanilla (80g),80g


In [0]:
delta_table.alias("t").merge(
    source = df_sb_dim_products.alias("s"), 
    condition = "t.product_code = s.product_code"
    ).whenMatchedUpdate(
        set={
            "division" : "s.division",
            "category" : "s.category",
            "product" : "s.product",
            "variant" : "s.variant"
        }
    ).whenNotMatchedInsert(
        values={
            "product_code" : "s.product_code",
            "division" : "s.division",
            "category" : "s.category",
            "product" : "s.product",
            "variant" : "s.variant"
        }
    ).execute()


DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
df_gross_price = DeltaTable.forName(spark, "fmcg.gold.dim_gross_price")df_sb_dim_gross_price = spark.sql(f"select* from {catalog}.{gold_schema}.sb_dim_gross_price")

df_sb_dim_gross_price.printSchema()

root
 |-- product_code: string (nullable = true)
 |-- month: date (nullable = true)
 |-- gross_price: double (nullable = true)



In [0]:
from pyspark.sql.window import Window

### Derive non-zero product gross price from latest month in an year using window function

In [0]:
df_sb_dim_gross_price = df_sb_dim_gross_price.withColumn(
    "year", F.year("month")
).withColumn(
    "is_zero", F.when(F.col("gross_price") == 0, 1).otherwise(0)
)

w = (Window
     .partitionBy("product_code","year")
     .orderBy(F.col("is_zero"), F.col("month").desc())     
     )

df_sb_dim_latest_gross_price = df_sb_dim_gross_price.withColumn(
    "rnk", F.row_number().over(w) 
).filter(F.col("rnk") == 1)

### Fetch and re-order columns as required

In [0]:
## Take required cols
df_sb_dim_latest_gross_price = df_sb_dim_latest_gross_price.select("product_code", "year", "gross_price").withColumnRenamed("gross_price", "price_inr").select("product_code", "price_inr", "year")

# change year to string
df_sb_dim_latest_gross_price = df_sb_dim_latest_gross_price.withColumn("year", F.col("year").cast("string"))

df_sb_dim_latest_gross_price.show(5)

+--------------------+---------+----+
|        product_code|price_inr|year|
+--------------------+---------+----+
|                NULL|     83.0|2025|
|062f5574bbdf4386b...|    281.0|2025|
|0cb7b2f42657b625f...|    100.0|2025|
|102628255d24304d6...|     86.0|2025|
|2e387cef1424d6e7b...|    108.0|2025|
+--------------------+---------+----+
only showing top 5 rows


In [0]:
df_sb_dim_latest_gross_price.printSchema()

root
 |-- product_code: string (nullable = true)
 |-- price_inr: double (nullable = true)
 |-- year: string (nullable = true)



In [0]:
delta_table = DeltaTable.forName(spark, "fmcg.gold.dim_gross_price")
print(delta_table.toDF().count())

794


%md
### Upsert in target table with more priority given to source table for gross price data

In [0]:
delta_table.alias("t").merge(
    source = df_sb_dim_latest_gross_price.alias("s"),
    condition = "t.product_code = s.product_code"
).whenMatchedUpdate(
    set={
        "price_inr" : "s.price_inr",
        "year" : "s.year"
    }
).whenNotMatchedInsert(
    values={
        "product_code" : "s.product_code",
        "price_inr" : "s.price_inr",
        "year" : "s.year"
    }
).execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
print(delta_table.toDF().count())

812
